# 02B Maximum Likelihood: Optimization and Applications

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/blob/main/06-Econometrics/02B_MLE_Optimization_and_Applications.ipynb) [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/main?filepath=06-Econometrics/02B_MLE_Optimization_and_Applications.ipynb) [![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)


In [ ]:
# === Environment Setup ===
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import Markdown, display
from scipy.optimize import minimize
from scipy.stats import norm

# --- Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (11, 7), 'figure.dpi': 130})
%config InlineBackend.figure_format = 'retina'
np.set_printoptions(suppress=True, linewidth=120, precision=4)


## The Lens: Solving MLE in Practice
**What problem are we solving?**
Analytical MLE solutions are rare in applied work. We need numerical optimization tools and workflows that scale to real-world models like Probit and Logit.

**Why this method?**
Numerical optimizers and reusable likelihood classes let us estimate complex models and then validate results with professional software.



**Economic question.** In *02B Maximum Likelihood: Optimization and Applications*, what must remain economically invariant when the computational representation changes? The computational task only has economic meaning after the estimand and identifying assumptions are explicit. Ask what variation identifies the parameter, which observations act as the comparison group, and what data-generating process would make the estimator fail. A good empirical workflow pairs the point estimate with diagnostics, uncertainty, and at least one falsification or sensitivity check so that precision is not confused with identification.

### Learning Objectives
* **Implement** numerical MLE with reusable optimization code.
* **Estimate** a Probit model from synthetic data and compare to statsmodels.
* **Visualize** likelihood surfaces and interpret hypothesis tests.

### Prerequisites
* **`06-Econometrics/02A_MLE_Principles_and_Geometry.ipynb`**: Likelihood fundamentals.
* **`02-Numerical-Methods/05_Optimization.ipynb`**: Optimization algorithms.
* **Probability:** PDFs, CDFs, and Normal distribution.


> **Learning path:** Building on [`02A_MLE_Principles_and_Geometry.ipynb`](02A_MLE_Principles_and_Geometry.ipynb); next continue with [`03_Causal_Inference.ipynb`](03_Causal_Inference.ipynb).


### Table of Contents
1. [The Lens: Solving MLE in Practice](#the-lens-solving-mle-in-practice)
2. [Numerical Optimization and Implementation](#1-numerical-optimization-and-implementation)
3. [A Reusable `MLEstimator` Class](#a-reusable-mlestimator-class)
4. [Application: Probit Model for Binary Choice](#2-application-probit-model-for-binary-choice)
5. [Verification with Statsmodels](#verification-with-statsmodels)
6. [Hypothesis Testing: The Holy Trinity](#5-hypothesis-testing-the-holy-trinity)
7. [Summary and Key Takeaways](#summary)


<a id='numerical'></a>
## 1. Numerical Optimization and Implementation

For most models (like Probit or Logit), we cannot find $\hat{\theta}$ analytically. We use numerical optimization (like the Newton-Raphson or BFGS algorithms) to climb the log-likelihood surface.

We will demonstrate MLE on a **Probit model**. A Probit model assumes a latent variable $y^* = X\beta + \epsilon$, where $\epsilon \sim N(0, 1)$. We observe $y=1$ if $y^* > 0$ and $y=0$ otherwise.

The probability of success is:
$$ P(y=1|X) = \Phi(X\beta) $$
where $\Phi$ is the standard normal CDF.

The log-likelihood contribution for observation $i$ is:
$$ \mathcal{L}_i(\beta) = y_i \ln \Phi(X_i\beta) + (1-y_i) \ln (1-\Phi(X_i\beta)) $$

**Dimension notes:** per observation, covariates $X_i \in \mathbb{R}^k$ and latent index $y_i^* = X_i'\beta + \epsilon_i$ scalar with $\beta \in \mathbb{R}^k$; observed outcome $y_i \in \{0, 1\}$; each log-likelihood contribution $\mathcal{L}_i(\beta)$ is a scalar, and the full objective sums over the $n$ observations.

<a id='mle-class'></a>
### A Reusable `MLEstimator` Class

To keep our code modular, we encapsulate the log-likelihood and optimization steps in a reusable class. This lets us swap in different models by changing only the log-likelihood function.


In [ ]:
class MLEstimator:
    """
    A class to perform Maximum Likelihood Estimation for a given model.

    This class is designed to be a general-purpose tool for estimating parameters
    of any model for which a log-likelihood function can be specified.
    """

    def __init__(self, loglike_func, data, param_names=None):
        """
        Initializes the MLEstimator.

        Parameters
        ----------
        loglike_func : callable
            The log-likelihood function. Must take two arguments: `params` (a
            NumPy array of parameters) and `data` (the data used for estimation).
            It should return the total log-likelihood value.
        data : object
            The data to be used in estimation. The format is flexible and should
            be handled by the user-provided loglike_func.
        param_names : list of str, optional
            A list of names for the parameters being estimated. If None, generic
            names like 'theta_0', 'theta_1', etc., will be used.
        """
        self.loglike = loglike_func
        self.data = data
        self.param_names = param_names
        self.results = None

    def fit(self, start_params):
        """
        Fit the model using a numerical optimizer to find the MLE.

        Parameters
        ----------
        start_params : np.ndarray
            An array of starting values for the optimization. The length must
            match the number of parameters.

        Returns
        -------
        self
            Returns the instance of the estimator.
        """
        if self.param_names is None:
            self.param_names = [f"theta_{i}" for i in range(len(start_params))]

        # The objective function is the *negative* of the log-likelihood,
        # because scipy.optimize performs minimization.
        def objective(params):
            return -self.loglike(params, self.data)

        # Use the BFGS algorithm to find the minimum of the negative log-likelihood
        # BFGS supplies an approximate inverse Hessian of the negative log-likelihood
        res = minimize(objective, start_params, method="BFGS", options={"disp": False})

        if not res.success or not np.isfinite(res.fun) or not np.all(np.isfinite(res.x)):
            raise RuntimeError(f"MLE optimization failed: {res.message}")

        # Store results only after successful convergence.
        self.mle_params = res.x
        # This optimizer approximation is illustrative, not a robust covariance.
        # Compare with observed-information SEs from specialist software.
        self.vcov = res.hess_inv
        self.std_errs = np.sqrt(np.diag(self.vcov))
        self.loglike_val = -res.fun
        self.results = res
        return self

    def summary(self):
        """
        Display a summary table of the estimation results, similar to those
        produced by standard econometric software.
        """
        if self.results is None:
            print("Model has not been fitted yet.")
            return

        # Calculate z-scores and p-values for hypothesis tests
        z_scores = self.mle_params / self.std_errs
        p_values = norm.sf(np.abs(z_scores)) * 2

        # Calculate 95% confidence intervals
        ci_lower = self.mle_params - 1.96 * self.std_errs
        ci_upper = self.mle_params + 1.96 * self.std_errs

        # Create a pandas DataFrame for a nicely formatted table
        summary_df = pd.DataFrame(
            {
                "Coefficient": self.mle_params,
                "Std. Error": self.std_errs,
                "z-score": z_scores,
                "p-value": p_values,
                "[0.025": ci_lower,
                "0.975]": ci_upper,
            },
            index=self.param_names,
        )

        print(f"Maximum Log-Likelihood: {self.loglike_val:.4f}")
        try:
            # Try to infer N from data structure
            if isinstance(self.data, dict):
                N = len(list(self.data.values())[0])
            else:
                N = len(self.data)
            print(f"Number of Observations: {N}")
        except (TypeError, AttributeError, IndexError):
            display(Markdown("> **Note:** Observation count is unavailable for this data container."))

        display(summary_df.round(4))
        return summary_df


<a id='probit'></a>
## 2. Application: Probit Model for Binary Choice

We now estimate a Probit model using synthetic data and the reusable MLE class.


In [ ]:
# 1. Generate Synthetic Data for Probit
rng = np.random.default_rng(seed=42)
N = 1000
X = sm.add_constant(rng.normal(0, 1, size=(N, 2))) # Constant, x1, x2
true_beta = np.array([-0.5, 1.2, -0.8]) # True parameters

# Latent variable y* = X*beta + e
latent_y = X @ true_beta + rng.normal(size=N)
y = (latent_y > 0).astype(int)

# 2. Define Log-Likelihood for Probit
def loglike_probit(beta, data):
    y, X = data['y'], data['X']
    # log Phi(z) for successes; log Phi(-z) for failures, stable in the tails.
    return np.sum(norm.logcdf((2 * y - 1) * (X @ beta)))

# 3. Estimate
data_probit = {'y': y, 'X': X}
mle_probit = MLEstimator(loglike_probit, data_probit, param_names=['Const', 'Beta1', 'Beta2'])
mle_probit.fit(start_params=[0, 0, 0])

print("Estimated Probit Model (Manual MLE):")
mle_probit.summary()


<a id='verify'></a>
### Verification with Statsmodels
Let's compare our manual implementation with the professional `statsmodels` library to ensure correctness.


In [ ]:
sm_model = sm.Probit(y, X)
sm_results = sm_model.fit(disp=0)
print(sm_results.summary())
np.testing.assert_allclose(mle_probit.mle_params, sm_results.params, atol=1e-6)
np.testing.assert_allclose(mle_probit.loglike_val, sm_results.llf, atol=1e-8)


## 5. Hypothesis Testing: The Holy Trinity

The figure shows a fixed-intercept slice of the log-likelihood and the restriction $H_0: \beta_1 = 0$. It does not calculate Wald, likelihood-ratio, or score tests. A likelihood-ratio test requires re-estimating all unrestricted nuisance parameters under the null; a fixed-intercept slice is not a profile likelihood.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

# Create grid for Beta1 vs Beta2 (holding Const fixed at MLE)
b_const = mle_probit.mle_params[0]
b1_vals = np.linspace(-0.1, 1.9, 50)
b2_vals = np.linspace(-1.5, -0.1, 50)
B1, B2 = np.meshgrid(b1_vals, b2_vals)
LL = np.zeros_like(B1)

for i in range(50):
    for j in range(50):
        # Calculate LL at this point
        LL[i, j] = loglike_probit([b_const, B1[i, j], B2[i, j]], data_probit)

# Contour plot
cs = ax.contour(B1, B2, LL, levels=20, cmap='viridis')
ax.clabel(cs, inline=1, fontsize=10)

# Mark MLE
ax.plot(mle_probit.mle_params[1], mle_probit.mle_params[2], 'r*', ms=15, label='Unrestricted MLE')

# Mark the null restriction, not a restricted MLE
ax.axvline(0, color='k', linestyle='--', label=r'Restriction $\beta_1=0$')

ax.set_title(r'Log-Likelihood Surface: $\mathcal{L}(\beta_1, \beta_2)$')
ax.set_xlabel('Beta 1')
ax.set_ylabel('Beta 2')
ax.legend()
plt.show()


## Exercises

**1. Mechanism and assumptions (Conceptual):** Define the estimand in **02B Maximum Likelihood: Optimization and Applications**, list the identifying assumptions, and give a concrete data-generating process that violates one assumption while leaving the others intact.

**2. Reproduce and diagnose (Applied):** Implement or reproduce the estimator using the material on 1. Numerical Optimization and Implementation, A Reusable `MLEstimator` Class. Report uncertainty and at least two diagnostics; then compare with an alternative specification that targets the same estimand.

**3. Robust extension (Challenge):** Run a Monte Carlo or sensitivity exercise that varies the most fragile identifying condition. Quantify bias/coverage or the range of estimates and state what evidence would change your substantive conclusion.

**3b. Failure analysis (Challenge):** The optimizer 'converges' with gradient norm $10^{-2}$, and the numeric Hessian at the solution is not negative definite. Diagnose false convergence (tolerances, scaling, analytic vs numeric derivatives), repair with tighter criteria and supplied gradients, and verify by restarting from the reported optimum.

<details>
<summary>Solution guidance</summary>

A strong solution states assumptions before computation, includes an independent diagnostic or limiting-case check, and interprets the result in the units of the economic problem. For the challenge, separate changes caused by the economic assumption from changes caused by numerical approximation or tuning.

</details>


# Summary

1.  **Likelihood Principle**: We estimate parameters by finding the values that maximize the probability of observing the data we actually saw.
2.  **Implementation**: We built a `MLEstimator` class that uses `scipy.optimize` to minimize the negative log-likelihood.
3.  **Flexibility**: The class estimates the Probit example here. Other likelihoods may need parameter bounds, different optimizers, and model-specific convergence checks.
4.  **Properties**: Under identification, correct specification, and suitable regularity conditions, MLE is consistent, asymptotically normal, and efficient. Boundary parameters and separation can invalidate this approximation.


## References & Further Reading

- Wooldridge, J. M. (2010). *Econometric Analysis of Cross Section and Panel Data* (2nd ed.). MIT Press.
- Angrist, J. D. & Pischke, J.-S. (2009). *Mostly Harmless Econometrics*. Princeton University Press.
- Imbens, G. W. & Rubin, D. B. (2015). *Causal Inference for Statistics, Social, and Biomedical Sciences*. Cambridge University Press.
